In [1]:
import psi4
import pandas as pd
import os
import numpy as np
from lps_uscf import lps_solver

In [31]:
csv_file = 'open_shell_15atoms_vs_uhf.csv'

if os.path.exists(csv_file):
    df = pd.read_csv(csv_file)
    print(f"-> Loaded existing results: {len(df)} rows found.")
else:
    df = pd.DataFrame()
    print("-> No existing file found. Starting fresh.")

-> Loaded existing results: 144 rows found.


In [32]:
TP = ['LDA_K_TF', 1.0]
LAMBDA = 0.111111
EXC = ['LDA_X', 0.0, 'LDA_C_VWN', 0.0]
FA = [True, 1.0]
DIIS = True
MAX_ITER = 15000
DAMPING = [0.99, 0.9, 0.00009]
# D_guess = [GUESS_A, GUESS_B]
D_guess = None
verbose=True

psi4.set_options({'basis': 'UGBS_S', 
                  'DFT_SPHERICAL_POINTS': 6, 
                  'DFT_RADIAL_POINTS': 1000})

mol = psi4.geometry("""
units bohr
0 3
S
symmetry c1
""")

E, Da, Db, iterations = lps_solver(MAX_ITER,TP,EXC,LAMBDA,mol,DAMPING,FA,D_guess,DIIS,verbose)
print('\nFinal SCF energy: %.4f Hartree' % E)

Number of basis functions:   31

Starting SCF iterations:

    Iter            Energy            Delta E         dRMS

SCF Iter  1:      4916.48420144    4.91648E+03    3.89692E+02
SCF Iter  2:      4805.27925502   -1.11205E+02    3.82441E+02
SCF Iter  3:      4696.25357334   -1.09026E+02    3.75321E+02
SCF Iter  4:      4597.87521992   -9.83784E+01    3.69877E+02
SCF Iter  5:      4501.27064076   -9.66046E+01    3.67670E+02
SCF Iter  6:      4406.54668257   -9.47240E+01    3.68649E+02
SCF Iter  7:      4313.69984647   -9.28468E+01    3.72681E+02
SCF Iter  8:      4220.48714825   -9.32127E+01    3.66427E+02
SCF Iter  9:      4129.78727782   -9.06999E+01    3.60262E+02
SCF Iter 10:      4042.04976984   -8.77375E+01    3.54043E+02
SCF Iter 11:      3956.33474037   -8.57150E+01    3.48018E+02
SCF Iter 12:      3878.42780872   -7.79069E+01    3.42271E+02
SCF Iter 13:      3799.28568089   -7.91421E+01    3.36792E+02
SCF Iter 14:      3721.57895352   -7.77067E+01    3.31035E+02
SCF Iter 15: 

In [33]:
GUESS_A = Da
GUESS_B = Db

In [34]:
psi4.core.set_output_file('output.dat', False)

ATOMS = {
    # Period 3 (Na-Ar)
    'Na': {'mult': 2},  # [Ne] 3s1
    'Mg': {'mult': 1},  # [Ne] 3s2
    'Al': {'mult': 2},  # [Ne] 3s2 3p1
    'Si': {'mult': 3},  # [Ne] 3s2 3p2 
    'P':  {'mult': 4},  # [Ne] 3s2 3p3 
    'S':  {'mult': 3},  # [Ne] 3s2 3p4
    # 'Cl': {'mult': 2},  # [Ne] 3s2 3p5
    # 'Ar': {'mult': 1},  # [Ne] 3s2 3p6
    # Period 4 (Selected)
    # 'K':  {'mult': 2},  # [Ar] 4s1
    # 'Ca': {'mult': 1},  # [Ar] 4s2
    # 'Cu': {'mult': 2},  # [Ar] 3d10 4s1 
    # 'Zn': {'mult': 1},  # [Ar] 3d10 4s2
    # 'Kr': {'mult': 1},  # [Ar] 3d10 4s2 4p6
}

METHOD = "TF0.111111W LDA"
TP = ['LDA_K_TF', 1.0]
LAMBDA = 0.111111
EXC = ['LDA_X', 1.0, 'LDA_C_VWN', 0.0]
# EXC = ['GGA_X_PBE', 1.0, 'LDA_C_VWN', 0.0]
FA = [False, 1.0]
DIIS = True
MAX_ITER = 25000
DAMPING = [0.99, 0.9, 0.00009]
D_guess = [GUESS_A, GUESS_B]
# D_guess = None
verbose=True

psi4.set_options({'basis': 'UGBS_S', 
                  'DFT_SPHERICAL_POINTS': 6, 
                  'DFT_RADIAL_POINTS': 1000})

for atom in ATOMS:
    
    if not df.empty:
        exists = df[
            (df['Atom'] == atom) & 
            (df['Method'] == METHOD) & 
            (df['Basis'] == psi4.core.get_global_option("BASIS"))
        ]
        if not exists.empty:
            print(f"Skipping {atom} (Already exists for {METHOD}/{psi4.core.get_global_option("BASIS")})")
            continue

    print(f"Calculating {atom} with {METHOD}...")
    MOL = psi4.geometry(f"0 {ATOMS[atom]['mult']}\n {atom}\nsymmetry c1")
    try:
        E, Da, Db, iterations = lps_solver(MAX_ITER,TP,EXC,LAMBDA,MOL,DAMPING,FA,D_guess,DIIS,verbose)
        if iterations >= MAX_ITER:
            print("  !!! SCF failed to converge (Max cycles exceeded).")
        else:
            print(f"Calculated Energy: {E:.4f} Hartree")
            row = {
                "Method": METHOD,
                "Atom": atom,
                "Basis": psi4.core.get_global_option("BASIS"),
                "Grid_Sph": psi4.core.get_global_option("DFT_SPHERICAL_POINTS"),
                "Grid_Rad": psi4.core.get_global_option("DFT_RADIAL_POINTS"),
                "Energy,Ha": round(E, 6),
                "Iterations": iterations,
                "DIIS": DIIS,
                "Damp_Start": DAMPING[0],
                "Damp_End": DAMPING[1],
                "Damp_Cutoff": DAMPING[2]
            }
            
            df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    
    except Exception as e:
        print(f"  !!! Failed {atom}. Error: {e}")
        continue

Skipping Na (Already exists for TF0.111111W LDA/UGBS_S)
Skipping Mg (Already exists for TF0.111111W LDA/UGBS_S)
Skipping Al (Already exists for TF0.111111W LDA/UGBS_S)
Skipping Si (Already exists for TF0.111111W LDA/UGBS_S)
Skipping P (Already exists for TF0.111111W LDA/UGBS_S)
Calculating S with TF0.111111W LDA...
Number of basis functions:   31

Starting SCF iterations:

    Iter            Energy            Delta E         dRMS

SCF Iter  1:      -424.99168312   -4.24992E+02    1.02316E+00
SCF Iter  2:      -425.02108804   -2.94049E-02    3.60165E-01
SCF Iter  3:      -425.02587433   -4.78630E-03    3.74996E-01
SCF Iter  4:      -424.93898872    8.68856E-02    5.58339E-01
SCF Iter  5:      -424.78658172    1.52407E-01    1.11946E+00
SCF Iter  6:      -424.55914192    2.27440E-01    1.71169E+00
SCF Iter  7:      -424.27432795    2.84814E-01    2.29612E+00
SCF Iter  8:      -423.89181510    3.82513E-01    2.86715E+00
SCF Iter  9:      -423.44974249    4.42073E-01    3.42381E+00
SCF It

KeyboardInterrupt: 

In [35]:
df.to_csv(csv_file, index=False)
df

,Method,Atom,Basis,Grid_Sph,Grid_Rad,"Energy,Ha",Iterations,DIIS,Damp_Start,Damp_End,Damp_Cutoff
0,TFW LDA,Na,UGBS_S,6,1000,-108.912083,284,True,0.90,0.90,0.00009
1,TFW LDA,Mg,UGBS_S,6,1000,-135.553609,237,True,0.90,0.90,0.00009
2,TFW LDA,Al,UGBS_S,6,1000,-165.669941,387,True,0.90,0.90,0.00009
3,TFW LDA,Si,UGBS_S,6,1000,-199.392286,517,True,0.90,0.90,0.00009
4,TFW LDA,P,UGBS_S,6,1000,-236.833553,342,True,0.90,0.90,0.00009
...,...,...,...,...,...,...,...,...,...,...,...
139,TF0.111111W LDA,Na,UGBS_S,6,1000,-175.220390,4870,True,0.99,0.99,0.00003
140,TF0.111111W LDA,Mg,UGBS_S,6,1000,-215.265035,4402,True,0.99,0.99,0.00003
141,TF0.111111W LDA,Al,UGBS_S,6,1000,-260.117429,14370,True,0.99,0.99,0.00003
142,TF0.111111W LDA,Si,UGBS_S,6,1000,-309.909768,8732,True,0.99,0.90,0.00005


In [ ]:
## TF0.111111W PBE for O and F used TF0.111111W FA densities of O and F as initial guesses
## TF0.111111W FA for Si, Kr used TF0.166666W FA initial guess
## TF0.2W FA for Cl used TF0.333333W FA initial guess
## TF0.166666W FA for Cl used TF0.2W FA initial guess
## TF0.111111W LDA for Si, P used TF0.111111W LDA density of a previous atom

In [ ]:
ATOMS = {
    'He':  {'mult': 1}, 
    'Be': {'mult': 1},
    'Ne': {'mult': 1}, 
    'Mg': {'mult': 1},
    'Ar':  {'mult': 1}, 
    'Ca':  {'mult': 1},
    'Zn':  {'mult': 1}, 
    'Kr':  {'mult': 1}
}
psi4.core.set_output_file('output.dat', False)
psi4.set_options({'basis': 'UGBS',
                  'scf_type': 'PK'})
rhf_energies = {}
rhf_homos = {}
for atom in ATOMS:
    MOL = psi4.geometry(f"0 {ATOMS[atom]['mult']}\n {atom}\nsymmetry c1")
    E, wfn = psi4.energy('SCF', return_wfn=True)
    rhf_energies[atom] = round(E, 6)
    print(f"RHF/UGBS {atom} Energy: {E:.4f} Hartree")

RHF/UGBS He Energy: -2.8617 Hartree, IP: -0.9180
RHF/UGBS Be Energy: -14.5730 Hartree, IP: -0.3093
RHF/UGBS Ne Energy: -128.5471 Hartree, IP: -0.8504
RHF/UGBS Mg Energy: -199.6146 Hartree, IP: -0.2530
RHF/UGBS Ar Energy: -526.8175 Hartree, IP: -0.5910
RHF/UGBS Ca Energy: -676.7582 Hartree, IP: -0.1955
RHF/UGBS Zn Energy: -1777.8481 Hartree, IP: -0.2925
RHF/UGBS Kr Energy: -2752.0549 Hartree, IP: -0.5242


In [26]:
atom_order = ['He', 'Be', 'Ne', 'Mg', 'Ar', 'Ca', 'Zn', 'Kr']
energy_table = df.pivot(index='Atom', columns='Method', values='Energy,Ha')
energy_table = energy_table.reindex(atom_order)
energy_table['RHF/UGBS'] = pd.Series(rhf_energies)
new_order = ['TFD0.166666W', 'TF0.166666W PBEx', 'TF0.166666W FA', 'RHF/UGBS']
energy_table = energy_table[new_order]

In [28]:
reference = energy_table['RHF/UGBS']

res = {}
for method, energies in energy_table.items():
    if method == 'RHF/UGBS': 
        continue 
    
    mae  = (energies - reference).abs().mean()
    rmae = (((energies - reference).abs()) / reference * -100).mean()
    res[method] = round(mae, 2), round(rmae, 2)

mae_row  = {method: values[0] for method, values in res.items()}
rmae_row = {method: values[1] for method, values in res.items()}
stats_df = pd.DataFrame([mae_row, rmae_row], index=['MAE(Ha)', 'rMAE(%)'])
energy_table = pd.concat([energy_table, stats_df], sort=False)

In [29]:
display(energy_table)

,TFD0.166666W,TF0.166666W PBEx,TF0.166666W FA,RHF/UGBS
He,-2.951247,-3.097069,-3.019526,-2.861680
Be,-15.040391,-15.388877,-14.660508,-14.573023
Ne,-132.508856,-133.573716,-128.291707,-128.547083
Mg,-204.540690,-205.864549,-198.305386,-199.614621
Ar,-537.208760,-539.346529,-522.928926,-526.817486
Ca,-690.396266,-692.814969,-672.808485,-676.758154
Zn,-1812.192490,-1816.067937,-1773.727670,-1777.848060
Kr,-2795.914635,-2800.696590,-2741.665221,-2752.054860
MAE(Ha),13.960000,15.970000,3.020000,NaN
rMAE(%),2.420000,3.690000,1.110000,NaN
